**Import Key Libraries**

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
from bs4 import BeautifulSoup
import lxml
import pandas as pd
import re
from datetime import datetime
from selenium.webdriver.common.keys import Keys 

### Step1: Get all product Links

In [ ]:
#Inputs to search
search_box_text = 'sports shoes for women'  # Change this to scrape other products (e.g., 'laptops', 'watches')
website_link = 'https://www.flipkart.com/'

#initiating the browser
#session start time
session_start_time = datetime.now().time()
print(f"Session Start Time: {session_start_time} ---------------------------> ")


#starting the browser with bot detection bypass
options = webdriver.ChromeOptions()
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

driver = webdriver.Chrome(options=options)
driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
    "source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
})

driver.get(website_link)
driver.maximize_window()

print('Waiting for search input...')
search_input = WebDriverWait(driver, 120).until(EC.presence_of_element_located((By.CSS_SELECTOR, '[autocomplete="off"]'))) 
        
print('Typing in search input...') 
search_input.send_keys(search_box_text) 
        
print('Submitting search form...') 
search_input.send_keys(Keys.RETURN) 
        
print('Waiting for search results...') 
WebDriverWait(driver, 120).until( EC.presence_of_element_located((By.CSS_SELECTOR, '[target="_blank"]')) )

print('Collecting pagination links...') 


#we want first 25 pages [pagination link]  [1000 Products]
#logic: Let's get the first page pagination link and append the number in the end for 25 pages and store in a list
all_pagination_links =[]

first_page = driver.find_elements(By.CSS_SELECTOR, 'nav a')[0]
first_page_link = first_page.get_attribute('href')
all_pagination_links.append(first_page_link)

for i in range(2, 26):
    new_pagination_link = first_page_link[: -1] + str(i)
    all_pagination_links.append(new_pagination_link)

print('Pagination Links Count:', len(all_pagination_links)) 
print("All Pagination Links: ", all_pagination_links)


print("Collecting Product Detail Page Links")
all_product_links = []

for link in all_pagination_links:
    driver.get(link)
    # Wait for the page to load by checking document.readyState
    WebDriverWait(driver, 120).until(lambda d: d.execute_script('return document.readyState') == 'complete')

    # Wait until elements are located (class name or product links containing /p/)
    WebDriverWait(driver, 120).until(EC.presence_of_element_located((By.CSS_SELECTOR, '.rPDeLR, a[href*="/p/"]'))) 
                
    all_products = driver.find_elements(By.CSS_SELECTOR, '.rPDeLR, a[href*="/p/"]')
    all_links = [element.get_attribute('href') for element in all_products]

    print(f"{link} Done ------>")

    all_product_links.extend(all_links)
    
print('All Product Detail Page Links Captured: ', len(all_product_links)) 


# Creating a DataFrame from the list
df_product_links = pd.DataFrame(all_product_links, columns=['product_links'])
#remove any duplicates
df_product_links = df_product_links.drop_duplicates(subset=['product_links'])

print("Total Product Detail Page Links", len(df_product_links))
df_product_links.to_csv('flipkart_product_links.csv', index = False)

driver.close()
session_end_time = datetime.now().time()
print(f"Session End Time: {session_end_time} ---------------------------> ")


### Step2: Get Individual product information

In [ ]:
#session start time
session_start_time = datetime.now().time()
print(f"Session Start Time: {session_start_time} ---------------------------> ")


#reading the csv file which contain all product links
df_product_links = pd.read_csv("flipkart_product_links.csv")

# Remove the below line to scrap all the products. For demonstration purpose we are scraping only 10 products
df_product_links = df_product_links.head(10)

all_product_links = df_product_links['product_links'].tolist()
print("Collecting Individual Product Detail Information")

#starting the browser with bot detection bypass
options = webdriver.ChromeOptions()
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

driver = webdriver.Chrome(options=options)
driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
    "source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
})


complete_product_details = []
unavailable_products = []
successful_parsed_urls_count = 0
complete_failed_urls_count = 0
for product_page_link in all_product_links:

    try: 
        driver.get(product_page_link)
    
        # Wait for the page to load by checking document.readyState
        WebDriverWait(driver, 120).until(lambda d: d.execute_script('return document.readyState') == 'complete')
    
        WebDriverWait(driver, 120).until( EC.presence_of_element_located((By.CSS_SELECTOR, '[target="_blank"], .VU-ZEz, .mEh187, .Nx9bqj')))
    
        #checking if product is available or not
        try:
            product_status = driver.find_element(By.CSS_SELECTOR, '.Z8JjpR, ._232Z7D').text
            if product_status in ['Currently Unavailable', 'Sold Out', 'Out of Stock']:
                unavailable_products.append(product_page_link)
                successful_parsed_urls_count += 1
                print(f"URL {successful_parsed_urls_count} completed --->")
                continue
        except:
            pass
    
        #brand with fallbacks
        try:
            brand = driver.find_element(By.CSS_SELECTOR, '.mEh187, span.B1GeXm, span.G6Xh1M').text.strip()
        except:
            brand = 'Generic'
    
        #title with fallbacks      
        try:
            title = driver.find_element(By.CSS_SELECTOR, '.VU-ZEz, h1.product-title').text.strip()
            title = re.sub(r'\s*\([^)]*\)', '', title)  #removing data within parenthesis
        except:
            title = 'Unknown Product'
    
        #price with fallbacks     
        try:
            price_text = driver.find_element(By.CSS_SELECTOR, '.Nx9bqj, div._30jeq3').text.strip()
            price = re.findall(r'\d+', price_text)
            price = ''.join(price)
        except:
            price = '0'
    
        # Discount with fallbacks 
        try:
            discount_text = driver.find_element(By.CSS_SELECTOR, '.UkUFwK, div._3Ay6B1').text.strip()
            discount = re.findall(r'\d+', discount_text)
            discount = ''.join(discount)
            discount = int(discount) / 100
        except:
            discount = ''
    
        # ratings with fallbacks   
        try:
            product_review_status = driver.find_element(By.CLASS_NAME, 'E3XX7J').text
            if product_review_status == 'Be the first to Review this product':
                avg_rating = ''
                total_ratings = ''
        except:
            try:
                avg_rating = driver.find_element(By.CSS_SELECTOR, '.XQDdHH, div._3LWZlK').text.strip()
                total_ratings_text = driver.find_element(By.CSS_SELECTOR, '.Wphh3N, span._2_R_DZ').text.split(' ')[0]
                if ',' in total_ratings_text:
                    total_ratings = int(total_ratings_text.replace(',', ''))
                else:
                    total_ratings = int(total_ratings_text)
            except:
                avg_rating = ''
                total_ratings = ''
    
        successful_parsed_urls_count += 1
        print(f"URL {successful_parsed_urls_count} completed *******")
        complete_product_details.append([product_page_link, title, brand, price, discount, avg_rating, total_ratings])  
    except Exception as e:
        print(f"Failed to establish a connection for URL {product_page_link}:  {e}")
        unavailable_products.append(product_page_link)
        complete_failed_urls_count += 1
        print(f"Failed URL Count {complete_failed_urls_count}")


#create pandas dataframe 
df = pd.DataFrame(complete_product_details, columns = ['product_link', 'title', 'brand', 'price', 'discount', 'avg_rating', 'total_ratings'])
#duplicates processing
df_duplicate_products = df[df.duplicated(subset=['brand', 'price', 'discount', 'avg_rating', 'total_ratings'])]
df = df.drop_duplicates(subset=['brand', 'price', 'discount', 'avg_rating', 'total_ratings'])
#unavailable products
df_unavailable_products = pd.DataFrame(unavailable_products, columns=['link'])


#prining the stats
print("Total product pages scrapped: ", len(all_product_links))
print("Final Total Products: ", len(df))
print("Total Unavailable Products : ", len(df_unavailable_products))
print("Total Duplicate Products: ", len(df_duplicate_products))


#saving all the files
df.to_csv('flipkart_product_data.csv', index = False)
df_unavailable_products.to_csv('unavailable_products.csv', index = False)
df_duplicate_products.to_csv('duplicate_products.csv', index = False)

# Only insert/overwrite SQLite database if we successfully scraped valid products.
# This acts as a fail-safe to preserve existing clean data if we get bot-blocked during live runs.
valid_scrapes = df[(df['title'] != 'Unknown Product') & (df['price'].astype(float) > 0)]
if not valid_scrapes.empty:
    import sqlite3
    try:
        conn = sqlite3.connect("../app/db.sqlite")
        cursor = conn.cursor()
        cursor.execute('''
        CREATE TABLE IF NOT EXISTS product (
            product_link TEXT,
            title TEXT,
            brand TEXT,
            price INTEGER,
            discount FLOAT,
            avg_rating FLOAT,
            total_ratings INTEGER
        );
        ''')
        conn.commit()
        
        # Clear existing rows to prevent duplicates on rerun
        cursor.execute("DELETE FROM product")
        conn.commit()
        
        # Clean whitespace from text columns before inserting
        for col in ['brand', 'title']:
            if col in df.columns:
                df[col] = df[col].astype(str).str.strip()
                
        df.to_sql('product', conn, if_exists='append', index=False)
        conn.close()
        print("Data inserted into SQLite database successfully!")
    except Exception as db_err:
        print(f"Failed to insert data into SQLite: {db_err}")
else:
    print("Scraper run returned only failed/blocked products. Skipped database overwrite to preserve existing records.")

driver.close()
session_end_time = datetime.now().time()
print(f"Session End Time: {session_end_time} ---------------------------> ")

Session Start Time: 17:11:34.372777 ---------------------------> 
URL 1 completed *******
URL 2 completed *******
URL 3 completed *******
URL 4 completed *******
URL 5 completed *******
URL 6 completed *******
URL 7 completed *******
URL 8 completed *******
URL 9 completed *******
URL 10 completed *******
Total product pages scrapped:  10
Final Total Products:  10
Total Unavailable Products :  0
Total Duplicate Products:  0
Session End Time: 17:11:49.935292 ---------------------------> 
